<div dir="rtl">

# 03 — إيه اللي جوّا الصندوق؟  ·  What's inside the box? A bilingual visual tour of a small transformer

> *في النوتبوك اللي فات (٠٢) شُفنا إزاي النص بيتقطّع لتوكنز. طب التوكنز دي بتروح فين بعد كده؟
> الموديل بيشتغل عليها إزاي وبيطلع تنبؤ منين؟ النوتبوك ده بياخدك في جولة جوّا موديل صغير
> (`pythia-160m`)، يسمّي كل قطعة جوّاه، ويوريك جملة ماصري وهي ماشية فيه من أول ما تدخل
> لحد ما الموديل يقول: "التوكن اللي جاي على الأرجح هو ده".*
>
> Notebook 02 showed how text becomes tokens. This one picks up where it left off:
> *what does the model actually do with those tokens?* We take a small transformer
> (`pythia-160m`), name every part of it, and follow a Masri prompt all the way through
> — from token IDs, to vectors, through twelve layers, to a probability over the next
> token. **No experiments yet, no interpretability claims.** Just naming the parts so
> we can do real work in nb05.

**Prereqs.** Notebook 02. Some Python. No prior ML required.  
**Companion (next).** [`04-logit-lens-masri.ipynb`](04-logit-lens-masri.ipynb) — once you know what a layer *is*, we'll watch the model's running guess evolve layer by layer.

**A note on the model.** `pythia-160m` is a 160-million-parameter English-trained model. We pick it because it's the canonical small target in mech interp tooling and TransformerLens supports it cleanly — not because it speaks Arabic well. From nb01 we know its tokenizer charges Arabic ~2x the English rate, and its predictions on Arabic will look weak. **That's fine for this notebook**, because our job is to name the box's parts, not to grade its outputs. Phase 2 is where output quality starts mattering.

</div>

<div dir="rtl">

## 1. الإعداد  ·  Setup

</div>

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import torch
from IPython.display import HTML, display
from transformer_lens import HookedTransformer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo:", REPO)
print("device:", DEVICE)

In [ ]:
# Load the model. HookedTransformer is the TransformerLens wrapper that exposes
# every internal activation as a named tensor we can inspect.
model = HookedTransformer.from_pretrained("pythia-160m", device=DEVICE)
model.eval()

print(f"layers (n_layers):        {model.cfg.n_layers}")
print(f"residual stream (d_model): {model.cfg.d_model}")
print(f"attention heads (n_heads): {model.cfg.n_heads}")
print(f"vocab size (d_vocab):      {model.cfg.d_vocab}")

<div dir="rtl">

أربع أرقام دول هما "شكل" الموديل ده. أي transformer صغير شكله كده تقريبًا: شوية طبقات، كل طبقة فيها رؤوس انتباه، ومتجه تمثيلي بحجم معين، ومفردات بحجم معين.

Those four numbers are the model's shape. Every small transformer looks roughly like this: some number of layers, each with attention heads, a representation vector of fixed width, and a vocabulary of fixed size. Keep them in your head — we'll touch each one below.

</div>

<div dir="rtl">

## 2. من النص للأرقام  ·  From text to token IDs (recap from nb02)

نكتب جملة ماصري قصيرة، نوريها للموديل. أول حاجة بتحصل: الـ tokenizer بيقطّعها لتوكنز ويحوّلها لأرقام (token IDs). ده اللي اتعلمناه في نوتبوك ٠٢ — بنسترجعه هنا في سطرين عشان نبدأ من نقطة معروفة.

We pick a short Masri sentence and feed it to the model. Step one is what nb02 already showed: the tokenizer chops it into pieces and converts each piece to an integer ID. We recap it here in two lines so the rest of the notebook starts from familiar ground.

</div>

In [ ]:
PROMPT = "عايز اروح"  # Masri: "I want to go"

tokens = model.to_tokens(PROMPT)  # shape: (batch=1, n_tokens)
token_strs = model.to_str_tokens(PROMPT)

print("prompt:    ", PROMPT)
print("token IDs: ", tokens.tolist()[0])
print("token strs:", token_strs)
print("shape:     ", tuple(tokens.shape), "  ← (batch, n_tokens)")

<div dir="rtl">

## 3. من الأرقام للمتجهات  ·  From numbers to vectors: the embedding

كل token ID هو رقم — لكن الموديل ما بيشتغلش على أرقام، بيشتغل على **متجهات** (vectors). أول حاجة الموديل بيعملها هي إنه بيدوّر كل token ID في جدول كبير اسمه **embedding matrix**، ويطلع متجه من `d_model` رقم. الجدول ده اتعلّم وقت تدريب الموديل، وهو فعليًا "معنى" كل توكن في عيون الموديل.

A token ID is just an integer — but the model doesn't compute on integers, it computes on **vectors**. The first thing it does is look each token ID up in a big table (the **embedding matrix**) and pull out a vector of `d_model` numbers. That table was learned during training, and it's effectively the model's "meaning" for each token.

</div>

In [ ]:
# The embedding matrix W_E has shape (d_vocab, d_model): one row per token in the vocab.
print("W_E shape:", tuple(model.W_E.shape), "  ← (d_vocab, d_model)")

# Embed our prompt manually so we can see what comes out.
with torch.no_grad():
    embeddings = model.W_E[tokens]  # (batch, n_tokens, d_model)

print("embeddings shape:", tuple(embeddings.shape), "  ← (batch, n_tokens, d_model)")
print()
print(f"first 8 dims of token 0 ({token_strs[0]!r}):")
print(embeddings[0, 0, :8].tolist())

<div dir="rtl">

## 4. الـ residual stream — العمود الفقري للموديل  ·  The residual stream: the model's spine

المتجه اللي طلع من الـ embedding ما بيتغيّرش لشكل تاني خالص لحد آخر الموديل. كل اللي بيحصل هو إن كل طبقة بتـ**ضيف** له شويّة معلومات. التنسور ده — شكله `(n_tokens, d_model)` — اسمه **the residual stream**، وهو زي العمود الفقري اللي كل طبقات الموديل بتعدّل عليه.

بمعنى تاني: الـ attention والـ MLP في كل طبقة مش بيـ"يحلّوا محل" المتجه، بس بيـ"يضيفوا" عليه. ده مهم عشان كده ممكن نبص للـ residual stream في أي طبقة ونفهم "إيه اللي الموديل عارفه عن كل توكن لحد الطبقة دي". في نوتبوك ٠٤ هنستغل ده مع الـ logit lens.

The vector that came out of the embedding doesn't get *replaced* on its way through the model — each layer just **adds** to it. That tensor, with shape `(n_tokens, d_model)`, is called the **residual stream**, and it's the spine every layer writes onto.

Said differently: the attention and MLP blocks in each layer don't overwrite the vector, they accumulate onto it. This is what makes it possible to look at the residual stream at any depth and ask "what does the model know about each token *so far*?" — which is exactly what nb04's logit lens will exploit.

</div>

<div dir="rtl">

## 5. الـ attention — التوكنز بتبصّ على بعض  ·  Attention: tokens look at each other

كل طبقة في الموديل فيها بلوكين: **attention** و **MLP**. الـ attention هو الجزء اللي بيخلّي كل توكن "يبصّ" على التوكنز اللي قبله ويلمّ منهم معلومات. مثلاً في جملة "عايز اروح"، التوكن "اروح" ممكن يحتاج يبصّ على "عايز" عشان يفهم إن السياق هنا فعل يتبعه فعل تاني.

كل طبقة فيها كذا "رأس" انتباه (`n_heads`)، كل واحد بيتعلّم نمط بصّ مختلف. مش هنحلل الأنماط دي هنا — بس هنشوف شكل التنسور ونتأكد إن أبعاده منطقية.

Each layer has two sub-blocks: **attention** and **MLP**. Attention is the part that lets each token *look at* the tokens before it and pull information from them. In "عايز اروح" ("I want to go"), the token for "اروح" might need to look at "عايز" to figure out it's in a verb-followed-by-verb context.

Each layer has multiple **heads** (`n_heads`), each learning a different attention pattern. We won't analyse the patterns here — just confirm the tensor shapes make sense.

</div>

In [ ]:
# Run the model and cache every internal activation we'll inspect.
with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

# Attention pattern at layer 0: shape (batch, n_heads, n_tokens, n_tokens).
# Entry [h, i, j] = how much head h at position i attends to position j.
attn_layer0 = cache["pattern", 0]
print("attention pattern (layer 0) shape:", tuple(attn_layer0.shape))
print("  ← (batch, n_heads, n_tokens, n_tokens)")
print()
print("head 0, attention from each token to each previous token:")
print(attn_layer0[0, 0].cpu().numpy().round(2))

<div dir="rtl">

## 6. الـ MLP — كل توكن بيفكّر لوحده  ·  MLP: each token thinks alone

بعد الـ attention، كل توكن بياخد لفّة جوّا شبكة عصبية صغيرة (الـ **MLP**)، بدون ما يبصّ على باقي التوكنز خالص. الجزء ده هو اللي بيـ"يحوّل" المعلومات اللي التوكن جمعها لشكل أنفع. لو الـ attention هو "التواصل"، الـ MLP هو "التفكير".

After attention, each token passes through a small feed-forward network (the **MLP**) on its own — no looking at other tokens. This is the part that *transforms* the information each token gathered into a more useful shape. If attention is "communication," MLP is "thinking."

</div>

In [ ]:
# MLP output at layer 0: shape (batch, n_tokens, d_model) — same shape as the
# residual stream, because that's what gets added back onto it.
mlp_out_layer0 = cache["mlp_out", 0]
print("mlp_out (layer 0) shape:", tuple(mlp_out_layer0.shape))
print("  ← (batch, n_tokens, d_model) — ready to be added onto the residual stream")

<div dir="rtl">

## 7. ١٢ طبقة فوق بعض  ·  Twelve layers, stacked

اللي شُفناه لحد دلوقتي (attention + MLP) ده محتوى **طبقة واحدة**. الموديل عنده ١٢ طبقة فوق بعض، وكل طبقة بتقرا الـ residual stream اللي طلع من اللي قبلها وتـضيف عليه. كل ما نروح أعمق، الـ residual stream بيبقى "عارف" أكتر عن السياق.

نطبع شكل الـ residual stream عند كل طبقة — الشكل بيفضل ثابت (`n_tokens, d_model`)، اللي بيتغيّر هو **محتوى** المتجهات.

Everything we've seen so far (attention + MLP) is the content of **one layer**. The model stacks twelve of them, each reading the residual stream from the previous layer and adding to it. The deeper we go, the more context the residual stream encodes.

We print the residual-stream shape at each layer — the shape stays the same (`n_tokens, d_model`), but the *contents* of the vectors change.

</div>

In [ ]:
for layer in range(model.cfg.n_layers):
    resid = cache["resid_post", layer]  # residual stream after layer's MLP
    print(f"layer {layer:2d}  resid_post shape: {tuple(resid.shape)}")

<div dir="rtl">

## 8. من الـ residual stream للتنبؤ  ·  From the final residual stream to a prediction

بعد آخر طبقة، الموديل عنده متجه من `d_model` رقم لكل توكن. ينفع منه يطلع تنبؤ إزاي؟ بيـ"يـضرب" المتجه ده في جدول تاني اسمه **unembedding matrix** (شكله `d_model × d_vocab`)، فيطلع رقم لكل كلمة محتملة في المفردات. الأرقام دي اسمها **logits**، ولما نطبّق عليها softmax بتبقى احتمالات.

لاحظ: الـ embedding في البداية كان `d_vocab → d_model`، والـ unembedding في الآخر هو `d_model → d_vocab`. الموديل بيدخل ويخرج من نفس "فضاء المفردات".

After the final layer, the model has a `d_model`-sized vector for each token. To turn that into a prediction, it multiplies by the **unembedding matrix** (shape `d_model × d_vocab`), producing one number per vocabulary entry. Those numbers are **logits**; applying softmax turns them into probabilities.

Notice the symmetry: embedding maps `d_vocab → d_model` on the way in; unembedding maps `d_model → d_vocab` on the way out. The model enters and exits the same vocabulary space.

</div>

In [ ]:
# logits shape: (batch, n_tokens, d_vocab) — a score for every vocab token
# at every position. The model's prediction for the *next* token after the
# whole prompt is at position -1.
print("logits shape:", tuple(logits.shape))

next_token_logits = logits[0, -1]
next_token_probs = torch.softmax(next_token_logits, dim=-1)

topk = torch.topk(next_token_probs, k=10)
print(f"\nTop-10 next-token guesses after {PROMPT!r}:")
for prob, tok_id in zip(topk.values.tolist(), topk.indices.tolist()):
    tok_str = model.to_string([tok_id])
    print(f"  {prob:6.2%}  {tok_id:>6}  {tok_str!r}")

<div dir="rtl">

*متوقع:* التنبؤات هتبقى ضعيفة لأن `pythia-160m` ما اتدرّبش على عربي تقريبًا. ده طبيعي ومش مشكلة — احنا هنا بنسمّي القطع، مش بنقيس الجودة.

*Expected:* the predictions will look weak because `pythia-160m` was barely trained on Arabic. That's fine — we're naming the parts, not grading the outputs.

</div>

<div dir="rtl">

## 9. الصورة كاملة  ·  The whole picture, end to end

نلمّ كل اللي اتعلمناه في صورة واحدة:

```
  "عايز اروح"
        ↓  (tokenizer, nb02)
  token IDs               shape: (n_tokens,)
        ↓  (embedding,  W_E)
  residual stream          shape: (n_tokens, d_model)
        ↓  layer 0  =  attention + MLP, added on
  residual stream          shape: (n_tokens, d_model)
        ↓  layer 1
        ⋮   (× 12 layers total)
        ↓  layer 11
  residual stream          shape: (n_tokens, d_model)
        ↓  (unembedding, W_U)
  logits                   shape: (n_tokens, d_vocab)
        ↓  softmax @ position -1
  probability over next token
```

كل قطعة في الصورة دي اتشرحت في قسم من الأقسام اللي فاتت. ده هو الموديل كله. مفيش حاجة تانية مخبية.

Every piece in this diagram was named in one of the sections above. That is the whole model. There is nothing else hidden inside.

</div>

<div dir="rtl">

## 10. خلاصة + الخطوة الجاية  ·  Recap and what's next

**اللي اتعلمناه:**

- الموديل بياخد توكنز → يحوّلهم لمتجهات (embedding) → يمشّيهم في residual stream عبر طبقات → كل طبقة فيها attention (التوكنز بتبصّ على بعض) و MLP (كل توكن بيفكّر لوحده) → آخر متجه يتحوّل لاحتمالات على المفردات (unembedding).
- الـ residual stream هو العمود الفقري. كل طبقة بتـ**ضيف** عليه، مش بتحلّ محله. ده اللي هيخلّي نوتبوك ٠٤ ممكن.
- الأشكال (shapes) دايمًا منطقية: لو متجه طلع بشكل غير اللي متوقعينه، يبقى فيه فهم غلط في مكان ما.

**الخطوة الجاية:** نوتبوك ٠٤ — الـ logit lens. هناخد الـ residual stream عند كل طبقة، نـ"نقرأه" كأنه آخر طبقة (نضربه في `W_U`)، ونشوف توقّع الموديل بيتطوّر إزاي مع العمق. ده أول حاجة تشبه "تجربة" interpretability حقيقية.

**Recap:**

- The model takes tokens → embeds them as vectors → flows them through a residual stream over layers → each layer has attention (tokens look at each other) and an MLP (each token thinks alone) → the final vector is unembedded into probabilities over the vocabulary.
- The residual stream is the spine. Every layer **adds** to it; nothing replaces it. That property is what makes nb04 possible.
- The shapes are always meaningful: if a tensor came out the wrong shape, your mental model is wrong somewhere.

**Next:** notebook 04 — the logit lens. We'll take the residual stream at every layer, *read* it as if it were the final layer (multiply by `W_U`), and watch the model's running guess evolve with depth. That's the first thing in this curriculum that resembles a real interpretability experiment.

</div>